# BTC price dynamics study

Answers, from data: **does BTC trend or mean-revert, at which horizon, and in which regime?**

**How to run:** Run All (Codespaces or any Jupyter). The first run downloads ~8 years of 1-minute BTCUSDT files from Binance's
free public archive (no API key) and caches them in the `data` folder, so later runs are fast.
To test everything without internet, set `SOURCE = "synthetic"` in the config cell.

**Reading the results:** every study ends with a table. The last cell prints one text summary of all of them -
paste it back to Claude to interpret.

**Limits:** one asset (BTCUSDT on Binance spot), thresholds are fixed placeholders (nothing is optimised),
and a pattern in the past is not a guarantee. Not financial advice.

## 1. Setup

In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
CACHE = "data"            # downloaded files are kept here, so later runs are fast

missing = []
for mod, pkg in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("pyarrow", "pyarrow")]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
if missing:   # a fresh Codespace needs all of these
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *missing])
    print("installed:", missing)

The library (data download, cleaning and all studies) is written to `btcstudy.py` by the next cell. You do not need to read it.

In [ ]:
%%writefile btcstudy.py
"""btcstudy - BTC data layer + price-dynamics studies.

Dependencies: pandas, numpy, requests, matplotlib (pyarrow optional, used for the cache).
Causality rule used everywhere: a signal/label computed from the close of bar t may only
be traded from bar t+1 on (see run_positions, which shifts positions by one bar).
"""
import hashlib
import io
import os
import sys
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

BASE = "https://data.binance.vision/data/spot"
KLINE_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
              "quote_volume", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]
KEEP = ["open", "high", "low", "close", "volume", "quote_volume", "trades", "taker_buy_base"]
OHLCV = ["open", "high", "low", "close", "volume"]

# validated palette (dataviz skill): categorical slots 1-3 + chrome
C = dict(s1="#2a78d6", s2="#eb6834", s3="#1baf7a", red="#e34948", ink="#0b0b0b", ink2="#52514e",
         muted="#898781", grid="#e1e0d9", axis="#c3c2b7", surface="#fcfcfb", mid="#f0efec")

try:
    import pyarrow  # noqa: F401
    _EXT = ".parquet"
except Exception:  # pragma: no cover
    _EXT = ".pkl"


# ----------------------------------------------------------------------------- cache helpers
def _save(df, base):
    if _EXT == ".parquet":
        try:
            df.to_parquet(base + ".parquet")
            return
        except Exception:
            pass
    df.to_pickle(base + ".pkl")


def _load(base):
    for ext in (".parquet", ".pkl"):
        p = base + ext
        if os.path.exists(p):
            try:
                return pd.read_parquet(p) if ext == ".parquet" else pd.read_pickle(p)
            except Exception:
                os.remove(p)  # corrupt cache entry -> refetch
    return None


# ----------------------------------------------------------------------------- Binance bulk data
def parse_kline_csv(f):
    """Parse one Binance kline CSV (file object). Handles header rows and ms/us timestamps."""
    raw = pd.read_csv(f, header=None, names=KLINE_COLS, usecols=range(12), dtype=str)
    if len(raw) and not str(raw.iloc[0, 0]).strip().isdigit():
        raw = raw.iloc[1:]  # a header line
    if len(raw) == 0:
        return pd.DataFrame(columns=KEEP, index=pd.DatetimeIndex([], tz="UTC", name="time"))
    ot = pd.to_numeric(raw["open_time"]).astype("int64")
    unit = "us" if ot.max() > 10 ** 14 else "ms"  # spot files use microseconds from 2025-01-01
    idx = pd.DatetimeIndex(pd.to_datetime(ot.to_numpy(), unit=unit, utc=True), name="time")
    if hasattr(idx, "as_unit"):
        idx = idx.as_unit("ns")
    out = raw[KEEP].apply(pd.to_numeric).astype("float64")
    out.index = idx
    return out


def parse_zip_bytes(b):
    with zipfile.ZipFile(io.BytesIO(b)) as z:
        with z.open(z.namelist()[0]) as f:
            return parse_kline_csv(f)


def kline_url(symbol, interval, period, kind):
    return f"{BASE}/{kind}/klines/{symbol}/{interval}/{symbol}-{interval}-{period}.zip"


def http_fetch(url, retries=4, timeout=90):
    """GET bytes; None if the file does not exist (404/403)."""
    import requests
    last = None
    for a in range(retries):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200:
                return r.content
            if r.status_code in (403, 404):
                return None
            last = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            last = repr(e)
        time.sleep(2 ** a)
    raise RuntimeError(f"download failed for {url}: {last}")


def _get_period(symbol, interval, kind, period, cache_dir, fetch, verify):
    base = os.path.join(cache_dir, f"{symbol}-{interval}-{period}")
    cached = _load(base)
    if cached is not None:
        return cached
    url = kline_url(symbol, interval, period, kind)
    data = fetch(url)
    if data is None:
        return None
    if verify:
        cs = fetch(url + ".CHECKSUM")
        if cs is not None:
            want = cs.decode().split()[0].lower()
            if want != hashlib.sha256(data).hexdigest():
                raise ValueError(f"checksum mismatch for {url}")
    df = parse_zip_bytes(data)
    _save(df, base)
    return df


def download_binance(symbol="BTCUSDT", interval="1m", start="2018-01-01", end=None, cache_dir="data",
                     workers=6, fetch=None, verify=True, verbose=True):
    """Download klines from data.binance.vision (free, no API key). Monthly files for complete
    months, daily files for the current month or when a monthly file is not published yet.
    `end` is exclusive and capped at today (UTC); only complete days are used."""
    fetch = fetch or http_fetch
    os.makedirs(cache_dir, exist_ok=True)
    today = pd.Timestamp.now("UTC").tz_localize(None).normalize()
    start = pd.Timestamp(start).normalize()
    end = min(pd.Timestamp(end).normalize(), today) if end else today
    if end <= start:
        raise ValueError("end must be after start")
    last_day = end - pd.Timedelta(days=1)
    months = pd.period_range(start, last_day, freq="M")

    def one(m):
        m_start, m_end = m.start_time.normalize(), m.end_time.normalize()
        if m_end < today:  # a finished month: try the single monthly file first, trim afterwards
            df = _get_period(symbol, interval, "monthly", str(m), cache_dir, fetch, verify)
            if df is not None:
                return df
        days = pd.date_range(max(m_start, start), min(m_end, last_day))
        parts = [_get_period(symbol, interval, "daily", d.strftime("%Y-%m-%d"), cache_dir, fetch, verify)
                 for d in days]
        parts = [p for p in parts if p is not None]
        return pd.concat(parts) if parts else None

    with ThreadPoolExecutor(workers) as ex:
        results = list(ex.map(one, months))
    got = [r for r in results if r is not None]
    if verbose:
        print(f"{symbol} {interval}: {len(got)}/{len(months)} months obtained")
    if not got:
        raise RuntimeError("no data found - check symbol, dates and internet access")
    df = pd.concat(got).sort_index()
    df = df[~df.index.duplicated(keep="first")]
    lo, hi = start.tz_localize("UTC"), end.tz_localize("UTC")
    return df[(df.index >= lo) & (df.index < hi)]


# ----------------------------------------------------------------------------- other sources
def load_custom_csv(path, ts_col="timestamp", rename=None):
    """Any CSV with a timestamp and open/high/low/close/volume columns (e.g. Tardis, CoinAPI exports)."""
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    if rename:
        df = df.rename(columns=rename)
    ts = df[ts_col]
    if pd.api.types.is_numeric_dtype(ts):
        m = float(ts.abs().max())
        unit = "ns" if m > 1e17 else "us" if m > 1e14 else "ms" if m > 1e11 else "s"
        idx = pd.to_datetime(ts, unit=unit, utc=True)
    else:
        idx = pd.to_datetime(ts, utc=True)
    df.index = pd.DatetimeIndex(idx, name="time")
    if "volume" not in df:
        df["volume"] = 0.0
    keep = [c for c in KEEP if c in df.columns]
    return df[keep].astype("float64").sort_index()


def ticks_to_ohlcv(ticks, ts_col="timestamp", price_col="price", size_col="amount", side_col=None, rule="1min"):
    """Aggregate trade-level data (e.g. a Tardis trades CSV) into OHLCV bars.
    If side_col is given (values 'buy'/'sell' = taker side) taker_buy_base is added."""
    t = ticks.copy()
    ts = t[ts_col]
    if pd.api.types.is_numeric_dtype(ts):
        m = float(ts.abs().max())
        unit = "ns" if m > 1e17 else "us" if m > 1e14 else "ms" if m > 1e11 else "s"
        idx = pd.to_datetime(ts, unit=unit, utc=True)
    else:
        idx = pd.to_datetime(ts, utc=True)
    t.index = pd.DatetimeIndex(idx)
    t = t.sort_index()
    g = t[price_col].resample(rule, closed="left", label="left")
    out = pd.DataFrame({"open": g.first(), "high": g.max(), "low": g.min(), "close": g.last(),
                        "volume": t[size_col].resample(rule, closed="left", label="left").sum(),
                        "trades": t[size_col].resample(rule, closed="left", label="left").count()})
    if side_col:
        buy = t[size_col].where(t[side_col].astype(str).str.lower() == "buy", 0.0)
        out["taker_buy_base"] = buy.resample(rule, closed="left", label="left").sum()
    out = out[out["open"].notna()]
    return out


def synthetic_1m(start="2022-01-01", days=900, seed=0):
    """Random-walk 1m data with volatility regimes and NO exploitable structure. For dry runs and tests."""
    rng = np.random.default_rng(seed)
    n = days * 1440
    ar = np.zeros(days)
    eps = rng.normal(0, 0.15, days)
    for i in range(1, days):
        ar[i] = 0.97 * ar[i - 1] + eps[i]
    sig = np.repeat(0.0006 * np.exp(0.5 * np.clip(ar, -2, 2)), 1440)
    r = rng.standard_normal(n) * sig
    close = 20000 * np.exp(np.cumsum(r))
    open_ = np.r_[close[0], close[:-1]]
    sp = np.abs(rng.standard_normal(n)) * sig * 0.5
    idx = pd.date_range(start, periods=n, freq="1min", tz="UTC", name="time")
    vol = rng.lognormal(1, 0.7, n) * (1 + 50 * sig)
    return pd.DataFrame({"open": open_, "high": np.maximum(open_, close) * (1 + sp),
                         "low": np.minimum(open_, close) * (1 - sp), "close": close, "volume": vol,
                         "quote_volume": vol * close, "trades": np.ceil(vol * 3),
                         "taker_buy_base": vol * 0.5}, index=idx)


def get_data(source="binance", **kw):
    """source: 'binance' (free bulk files) | 'csv' (load_custom_csv) | 'synthetic' (dry run)."""
    if source == "binance":
        df = download_binance(**kw)
    elif source == "csv":
        df = load_custom_csv(**kw)
    elif source == "synthetic":
        df = synthetic_1m(**kw)
    else:
        raise ValueError(f"unknown source {source}")
    return clean(df)


# ----------------------------------------------------------------------------- cleaning & quality
def _bad_ohlc(df):
    return ((df.high < df.low) | (df.close > df.high) | (df.close < df.low) |
            (df.open > df.high) | (df.open < df.low) | (df[["open", "high", "low", "close"]] <= 0).any(axis=1))


def clean(df):
    """Sort, drop duplicate timestamps and invalid OHLC rows. Gaps are NOT filled."""
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="first")]
    return df[~_bad_ohlc(df)]


def quality_report(df, freq="1min"):
    idx = df.index
    gaps = idx.to_series().diff().dropna()
    step = pd.Timedelta(freq)
    expected = int((idx[-1] - idx[0]) / step) + 1
    big = gaps[gaps > step]
    ret = np.log(df.close).diff()
    return {"start": str(idx[0]), "end": str(idx[-1]), "bars": len(df), "expected_bars": expected,
            "missing_bars": expected - len(df), "missing_pct": round(100 * (expected - len(df)) / expected, 4),
            "gaps": int(len(big)), "largest_gap": str(gaps.max()) if len(gaps) else "0",
            "zero_volume_bars": int((df.volume == 0).sum()), "bars_move_gt_5pct": int((ret.abs() > 0.05).sum()),
            "bad_ohlc_rows": int(_bad_ohlc(df).sum()),
            "top_gaps": [(str(i), str(g)) for i, g in big.nlargest(5).items()]}


def resample_ohlcv(df, rule):
    """Aggregate to a coarser bar. Bars are labelled by their OPEN time; empty bars are dropped."""
    agg = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
    for c in ("quote_volume", "trades", "taker_buy_base"):
        if c in df.columns:
            agg[c] = "sum"
    out = df.resample(rule, closed="left", label="left").agg(agg)
    return out[out["open"].notna()]


def build_frames(df1m):
    return {"1m": df1m, "h1": resample_ohlcv(df1m, "1h"), "h4": resample_ohlcv(df1m, "4h"),
            "daily": resample_ohlcv(df1m, "1D")}


# ----------------------------------------------------------------------------- study A: horizons
def lag1(x):
    x = np.asarray(x, float)
    if len(x) < 30:
        return np.nan
    d = x - x.mean()
    den = (d ** 2).sum()
    return float((d[1:] * d[:-1]).sum() / den) if den > 0 else np.nan


def autocorr_by_horizon(df1m, horizons=("1min", "5min", "15min", "1h", "4h", "1D")):
    """Lag-1 autocorrelation of non-overlapping log returns, per horizon and per calendar year.
    Positive = trend/momentum, negative = mean reversion."""
    rows = {}
    for h in horizons:
        c = df1m["close"] if h == "1min" else resample_ohlcv(df1m[OHLCV], h)["close"]
        r = np.log(c).diff().dropna()
        row = {int(y): lag1(g.values) for y, g in r.groupby(r.index.year)}
        row["all"] = lag1(r.values)
        row["n_all"] = len(r)
        row["2se"] = 2 / np.sqrt(len(r))
        rows[h] = row
    tab = pd.DataFrame(rows).T
    cols = [c for c in tab.columns if c not in ("all", "n_all", "2se")] + ["all", "n_all", "2se"]
    return tab[cols]


def variance_ratio(r, q):
    """Lo-MacKinlay variance ratio with heteroskedasticity-robust z. VR<1 reversion, VR>1 trend."""
    r = np.asarray(r, float)
    T = len(r)
    mu = r.mean()
    d = r - mu
    s_a = (d ** 2).sum() / (T - 1)
    rq = pd.Series(r).rolling(q).sum().dropna().to_numpy()
    m = q * (T - q + 1) * (1 - q / T)
    s_c = ((rq - q * mu) ** 2).sum() / m
    vr = s_c / s_a
    d2 = d ** 2
    den = d2.sum() ** 2
    theta = sum((2 * (q - j) / q) ** 2 * (T * (d2[j:] * d2[:-j]).sum() / den) for j in range(1, q))
    return float(vr), float(np.sqrt(T) * (vr - 1) / np.sqrt(theta))


def variance_ratio_table(df1m, specs=None):
    specs = specs or {"1h": (2, 4, 8, 24), "1D": (2, 5, 10, 20)}
    rows = []
    for bar, qs in specs.items():
        r = np.log(resample_ohlcv(df1m[OHLCV], bar)["close"]).diff().dropna()
        for q in qs:
            vr, z = variance_ratio(r.values, q)
            row = {"bar": bar, "q": q, "VR": vr, "z": z}
            for y, g in r.groupby(r.index.year):
                if len(g) > 4 * q + 30:
                    row[int(y)] = variance_ratio(g.values, q)[0]
            rows.append(row)
    return pd.DataFrame(rows).set_index(["bar", "q"])


# ----------------------------------------------------------------------------- study B: overshoots
def reversal_buckets(close, window=500, edges=(-np.inf, -3, -2, -1, 1, 2, 3, np.inf), cost_bps=5.0):
    """Next-bar return after a move of z standard deviations (z uses PAST volatility only).
    reversal_bps > 0 means the move reversed; net_bps subtracts a round trip (2 x cost_bps)."""
    r = np.log(close).diff()
    z = r / r.rolling(window).std().shift(1)
    d = pd.DataFrame({"z": z, "nxt": r.shift(-1)}).dropna()
    labels = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        labels.append(f"<{hi:g}" if np.isinf(lo) else f">{lo:g}" if np.isinf(hi) else f"{lo:g}..{hi:g}")
    d["b"] = pd.cut(d["z"], list(edges), labels=labels)
    g = d.groupby("b", observed=True)["nxt"]
    out = pd.DataFrame({"n": g.size(), "mean_next_bps": g.mean() * 1e4,
                        "t": g.mean() / (g.std() / np.sqrt(g.size())), "up_rate": g.apply(lambda s: (s > 0).mean())})
    lohi = dict(zip(labels, zip(edges[:-1], edges[1:])))
    rev = []
    for b, m in zip(out.index, out["mean_next_bps"]):
        lo, hi = lohi[b]
        rev.append(m if hi <= 0 else -m if lo >= 0 else np.nan)
    out["reversal_bps"] = rev
    out["net_bps"] = out["reversal_bps"] - 2 * cost_bps
    return out


def vol_persistence(daily_close):
    r = np.log(daily_close).diff().dropna()
    a = r.abs()
    res = {f"abs_return_acf_lag{k}": float(a.autocorr(k)) for k in (1, 5, 10, 30)}
    rv = r.rolling(20).std()
    med = rv.rolling(365, min_periods=180).median().shift(1)
    h = (rv > med).where(med.notna()).dropna().astype(bool)
    # horizons >= 20 days only: rv uses a 20-day window, so shorter horizons overlap and look persistent on pure noise.
    # Baseline for no persistence is about 0.5 in both rows.
    for k in (20, 60, 120):
        fut = h.shift(-k)
        ok = fut.notna()
        res[f"P(high vol in {k}d | high now)"] = float(fut[h & ok].astype(float).mean())
        res[f"P(high vol in {k}d | low now)"] = float(fut[~h & ok].astype(float).mean())
    res["share_of_days_high_vol"] = float(h.mean())
    return res


# ----------------------------------------------------------------------------- study C: regimes
def adx(df, n=14):
    h, l, c = df.high, df.low, df.close
    up, dn = h.diff(), -l.diff()
    pdm = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=df.index)
    mdm = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=df.index)
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1 / n, adjust=False).mean()
    pdi = 100 * pdm.ewm(alpha=1 / n, adjust=False).mean() / atr
    mdi = 100 * mdm.ewm(alpha=1 / n, adjust=False).mean() / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi)
    return dx.ewm(alpha=1 / n, adjust=False).mean(), atr


def _causal_rank(x, window):
    def f(w):
        past, cur = w[:-1], w[-1]
        past = past[~np.isnan(past)]
        if len(past) < 30 or np.isnan(cur):
            return np.nan
        return (past < cur).mean()
    return x.rolling(window, min_periods=window // 2).apply(f, raw=True)


def regime_labels(daily, lookback=28, pct_window=365, er_n=20, adx_n=14, adx_level=25.0):
    """All labels are computed from data up to the CLOSE of each daily bar (usable from the next bar).
      dir_mom : up / down / range  - lookback-day return in top/bottom/middle third of its trailing history
      er      : Kaufman efficiency ratio; er_state trend if above its trailing-year median
      adx     : Wilder ADX; adx_state trend if above adx_level
      vol_state: high / low - ATR% vs its trailing-year median"""
    c = daily["close"]
    ret_n = c / c.shift(lookback) - 1
    rank = _causal_rank(ret_n, pct_window)
    dir_mom = pd.Series(np.where(rank >= 2 / 3, "up", np.where(rank <= 1 / 3, "down", "range")),
                        index=daily.index).where(rank.notna())
    er = (c - c.shift(er_n)).abs() / c.diff().abs().rolling(er_n).sum()
    er_med = er.rolling(pct_window, min_periods=pct_window // 2).median().shift(1)
    er_state = pd.Series(np.where(er > er_med, "trend", "range"), index=daily.index).where(er_med.notna() & er.notna())
    a, atr = adx(daily, adx_n)
    adx_state = pd.Series(np.where(a > adx_level, "trend", "range"), index=daily.index).where(a.notna())
    atrp = atr / c
    v_med = atrp.rolling(pct_window, min_periods=pct_window // 2).median().shift(1)
    vol_state = pd.Series(np.where(atrp > v_med, "high", "low"), index=daily.index).where(v_med.notna())
    return pd.DataFrame({"ret_n": ret_n, "dir_mom": dir_mom, "er": er, "er_state": er_state,
                         "adx": a, "adx_state": adx_state, "vol_state": vol_state})


def evaluate_regimes(daily, labels, horizon=5):
    """Per classifier state: forward return, forward absolute move, TREND PAYOFF and persistence.
    trend payoff = forward return x sign(lookback return): >0 the prior direction continued, <0 it reversed."""
    c = daily["close"]
    fwd = c.shift(-horizon) / c - 1
    tf = np.sign(labels["ret_n"]) * fwd
    rows = []
    for col in ("dir_mom", "er_state", "adx_state", "vol_state"):
        nxt = labels[col].shift(-horizon)
        for st, idx in labels.groupby(col).groups.items():
            f = fwd.loc[idx].dropna()
            ok = nxt.loc[idx].notna()
            pers = (nxt.loc[idx][ok] == labels[col].loc[idx][ok]).mean() if ok.any() else np.nan
            rows.append({"classifier": col, "state": st, "days": len(f), "fwd_ret_pct": 100 * f.mean(),
                         "fwd_abs_move_pct": 100 * f.abs().mean(), "trend_payoff_pct": 100 * tf.loc[idx].dropna().mean(),
                         "persistence": pers})
    return pd.DataFrame(rows).set_index(["classifier", "state"])


def regime_consistency(daily, labels, col="dir_mom", horizon=5):
    """Trend payoff (%) by state and calendar year - does the regime effect hold in every year?"""
    c = daily["close"]
    tf = 100 * np.sign(labels["ret_n"]) * (c.shift(-horizon) / c - 1)
    d = pd.DataFrame({"s": labels[col], "tf": tf}).dropna()
    return d.groupby([d["s"], d.index.year])["tf"].mean().unstack(0).round(3)


# ----------------------------------------------------------------------------- study D: strategies
def perf(net, ppy):
    net = net.dropna()
    eq = (1 + net).cumprod()
    ann, vol = net.mean() * ppy, net.std() * np.sqrt(ppy)
    return {"total_ret": float(eq.iloc[-1] - 1), "ann_ret": float(ann), "ann_vol": float(vol),
            "sharpe": float(ann / vol) if vol > 0 else np.nan, "max_dd": float((eq / eq.cummax() - 1).min())}


def run_positions(close, pos_decided, cost_bps=5.0, ppy=365):
    """pos_decided[t] is decided at the CLOSE of bar t and applied to bar t+1. cost_bps is per side."""
    ret = close.pct_change()
    pos = pos_decided.shift(1).fillna(0.0)
    turn = pos.diff().abs().fillna(pos.abs())
    net = pos * ret - turn * cost_bps / 1e4
    st = perf(net, ppy)
    entries = int(((pos != 0) & (pos.shift(1).fillna(0) == 0)).sum())
    st.update({"exposure": float(pos.abs().mean()), "entries": entries,
               "avg_hold_bars": float(pos.abs().sum() / max(entries, 1))})
    return net, st


def pos_max(daily, n):
    c = daily["close"]
    return (c >= c.rolling(n).max()).astype(float)


def pos_min(daily, n):
    c = daily["close"]
    return (c <= c.rolling(n).min()).astype(float)


def pos_overshoot(close, k=2.5, hold=2, window=500, side="both"):
    r = np.log(close).diff()
    z = r / r.rolling(window).std().shift(1)
    L = (z < -k).astype(float).rolling(hold, min_periods=1).max()
    S = (z > k).astype(float).rolling(hold, min_periods=1).max()
    return {"long": L, "short": -S, "both": L - S}[side]


def strategy_suite(daily, hourly, labels, cost_bps=5.0):
    """Long-only trend rules (daily) and overshoot fades (hourly). Returns (stats table, net-return dict)."""
    strats = {"Buy & hold": (daily, pd.Series(1.0, index=daily.index), 365),
              "TS momentum 28d (top third)": (daily, (labels["dir_mom"] == "up").astype(float), 365)}
    for n in (10, 20, 50):
        strats[f"Buy new {n}d highs"] = (daily, pos_max(daily, n), 365)
        strats[f"Buy new {n}d lows"] = (daily, pos_min(daily, n), 365)
    for s in ("long", "short", "both"):
        strats[f"1h overshoot fade ({s})"] = (hourly, pos_overshoot(hourly["close"], side=s), 365 * 24)
    rows, nets = {}, {}
    for name, (bars, pos, ppy) in strats.items():
        net, st = run_positions(bars["close"], pos, cost_bps if name != "Buy & hold" else 0.0, ppy)
        rows[name], nets[name] = st, net
    return pd.DataFrame(rows).T, nets


def shuffled_days(bars, seed):
    """Luck surrogate: the same bars with WHOLE DAYS in random order. Each day keeps its intraday shape, the return
    distribution and the total drift are unchanged, but any day-to-day dependence (trend, reversal, vol clustering)
    is destroyed. Strategies that only work through such dependence should do no better here than by chance."""
    rng = np.random.default_rng(seed)
    close = bars["close"].to_numpy()
    r = np.diff(np.log(close), prepend=np.log(close[0]))
    codes = pd.factorize(bars.index.normalize())[0]
    cut = np.r_[0, np.flatnonzero(np.diff(codes)) + 1, len(bars)]
    blocks = [np.arange(a, b) for a, b in zip(cut[:-1], cut[1:])]
    order = np.r_[0, 1 + rng.permutation(len(blocks) - 1)]  # day 0 stays first: it has no overnight gap
    perm = np.concatenate([blocks[i] for i in order])
    pc = np.r_[bars["open"].iloc[0], close[:-1]]
    new_close = close[0] * np.exp(np.cumsum(r[perm]))
    new_pc = np.r_[close[0], new_close[:-1]]
    out = pd.DataFrame(index=bars.index)
    for c in ("open", "high", "low"):
        out[c] = new_pc * (bars[c].to_numpy() / pc)[perm]
    out["close"] = new_close
    out["volume"] = bars["volume"].to_numpy()[perm]
    return out[OHLCV]


def sign_flip_surrogate(bars, seed):
    """Luck surrogate (default): every bar's demeaned return gets a random +/- sign. The size of every move - and so
    volatility clustering and intraday seasonality - and the average drift are kept; all predictability of the
    DIRECTION (trend, reversal, momentum) is destroyed. Intrabar shape flips with the sign so OHLC stays valid."""
    rng = np.random.default_rng(seed)
    o, h, l, c = (bars[k].to_numpy(float) for k in ("open", "high", "low", "close"))
    pc = np.r_[o[0], c[:-1]]
    lc = np.log(c / pc)
    mu = lc.mean()
    s = rng.choice([-1.0, 1.0], size=len(bars))
    lo_, lh, ll = np.log(o / pc), np.log(h / pc), np.log(l / pc)
    new_close = o[0] * np.exp(np.cumsum(mu + s * (lc - mu)))
    new_pc = np.r_[o[0], new_close[:-1]]
    n_open = new_pc * np.exp(np.where(s > 0, lo_, -lo_))
    n_high = new_pc * np.exp(np.where(s > 0, lh, -ll))
    n_low = new_pc * np.exp(np.where(s > 0, ll, -lh))
    out = pd.DataFrame({"open": n_open, "high": np.maximum.reduce([n_high, n_open, new_close]),
                        "low": np.minimum.reduce([n_low, n_open, new_close]), "close": new_close,
                        "volume": bars["volume"].to_numpy()}, index=bars.index)
    return out[OHLCV]


def noise_comparison(h1, real_suite, n=40, cost_bps=5.0, seed=0, method="signflip"):
    """Real Sharpe of every strategy vs the Sharpe the same rules reach on `n` surrogate copies of the data
    (method 'signflip' = random return signs; 'days' = shuffled days, which proved miscalibrated on data with
    volatility clustering). p_value = chance a surrogate does at least as well (smallest possible 1/(n+1)).
    Calibration on 60 random-walk datasets with n=40: 5.5% of strategies flagged at the 95% level (nominal 5%).
    Several strategies are tested at once, so ask for p_value < 0.01, not 0.05, and consistency across years."""
    make = {"signflip": sign_flip_surrogate, "days": shuffled_days}[method]
    rows = []
    for k in range(n):
        s = make(h1[OHLCV], seed + k)
        d = resample_ohlcv(s, "1D")
        tab, _ = strategy_suite(d, s, regime_labels(d), cost_bps)
        rows.append(tab["sharpe"])
    noise = pd.DataFrame(rows)
    out = pd.DataFrame({"real_sharpe": real_suite["sharpe"], "luck_mean": noise.mean(), "luck_p95": noise.quantile(0.95)})
    out["p_value"] = (noise.ge(real_suite["sharpe"], axis=1).sum() + 1) / (len(noise) + 1)
    out["beats_luck_95"] = out["real_sharpe"] > out["luck_p95"]
    return out


def cost_sweep(daily, hourly, labels, costs=(0, 2, 5, 10, 20)):
    """Sharpe of each strategy at different per-side costs (bps)."""
    out = {}
    for c in costs:
        tab, _ = strategy_suite(daily, hourly, labels, cost_bps=float(c))
        out[f"{c} bps"] = tab["sharpe"]
    return pd.DataFrame(out)


def sharpe_by_year(net, ppy):
    net = net.dropna()
    return net.groupby(net.index.year).apply(lambda g: g.mean() * ppy / (g.std() * np.sqrt(ppy)) if g.std() > 0 else np.nan)


# ----------------------------------------------------------------------------- study E: zone strategy by regime
def zone_by_regime(df1m, labels, zsys_dir=None, tf="4h"):
    """Run the zone-confluence backtest (zsys package) and split its trades by the daily regime known
    at the previous close. Returns (trades, by_alignment, by_alignment_and_vol) or None."""
    if zsys_dir:
        sys.path.insert(0, zsys_dir)
    try:
        from zsys.zones import build, ZoneConfig
        from zsys.backtest import run, Params, Risk
    except ImportError:
        print("zsys not found - set ZSYS_DIR to the folder that contains the 'zsys' package")
        return None
    bars = resample_ohlcv(df1m, tf)[OHLCV].copy()
    bars.index = bars.index.tz_convert(None)
    tr, _ = run(bars, build(bars, ZoneConfig()), Params(), Risk())
    if len(tr) == 0:
        print("zone strategy produced no trades")
        return None
    lab = labels.copy()
    lab.index = lab.index.tz_convert(None) if lab.index.tz is not None else lab.index
    day = pd.to_datetime(tr["entry_time"]).dt.normalize() - pd.Timedelta(days=1)
    tr = tr.copy()
    tr["dir"] = lab["dir_mom"].reindex(day.values).to_numpy()
    tr["vol"] = lab["vol_state"].reindex(day.values).to_numpy()
    with_t = ((tr["dir"] == "up") & (tr["side"] == 1)) | ((tr["dir"] == "down") & (tr["side"] == -1))
    counter = ((tr["dir"] == "up") & (tr["side"] == -1)) | ((tr["dir"] == "down") & (tr["side"] == 1))
    tr["alignment"] = np.where(tr["dir"].isna(), "unknown", np.where(with_t, "with_trend",
                               np.where(counter, "counter_trend", "range")))

    def agg(g):
        return pd.Series({"trades": len(g), "win_rate": (g["pnl"] > 0).mean(), "exp_R": g["R"].mean(),
                          "total_R": g["R"].sum()})

    a = tr.groupby("alignment").apply(agg)
    b = tr.groupby(["alignment", "vol"]).apply(agg)
    return tr, a, b


# ----------------------------------------------------------------------------- display & charts
def show(obj, digits=3):
    try:
        obj = obj.round(digits)
    except Exception:
        pass
    try:
        from IPython.display import display
        display(obj)
    except Exception:
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


def _plt():
    import matplotlib.pyplot as plt
    return plt


def _style(ax, ylabel=None):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(C["axis"])
    ax.tick_params(colors=C["muted"], labelsize=9)
    ax.grid(axis="y", color=C["grid"], linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_facecolor(C["surface"])
    if ylabel:
        ax.set_ylabel(ylabel, color=C["ink2"], fontsize=9)


def _title(ax, text):
    ax.set_title(text, loc="left", fontsize=11, color=C["ink"], pad=10)


def plot_autocorr_heatmap(tab, show_fig=True):
    from matplotlib.colors import LinearSegmentedColormap
    plt = _plt()
    cols = [c for c in tab.columns if c not in ("n_all", "2se")]
    v = tab[cols].astype(float)
    lim = max(float(np.nanmax(np.abs(v.values))), 0.02)
    fig, ax = plt.subplots(figsize=(8, 0.55 * len(v) + 1.6), facecolor=C["surface"])
    im = ax.imshow(v.values, aspect="auto", vmin=-lim, vmax=lim,
                   cmap=LinearSegmentedColormap.from_list("d", [C["red"], C["mid"], C["s1"]]))
    ax.set_xticks(range(len(cols)), [str(c) for c in cols])
    ax.set_yticks(range(len(v)), list(v.index))
    ax.tick_params(colors=C["muted"], labelsize=9, length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    _title(ax, "Lag-1 autocorrelation by horizon and year  (blue = trend, red = reversal)")
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.ax.tick_params(colors=C["muted"], labelsize=8)
    fig.tight_layout()
    if show_fig:
        plt.show()
    return fig


def plot_reversal(bk, title, show_fig=True):
    plt = _plt()
    fig, ax = plt.subplots(figsize=(8, 3.6), facecolor=C["surface"])
    x = np.arange(len(bk))
    ax.bar(x, bk["mean_next_bps"], width=0.55, color=C["s1"])
    ax.axhline(0, color=C["axis"], linewidth=1)
    for xi, (val, n) in enumerate(zip(bk["mean_next_bps"], bk["n"])):
        ax.annotate(f"{val:.1f}", (xi, val), ha="center", va="bottom" if val >= 0 else "top",
                    fontsize=8, color=C["ink2"], xytext=(0, 3 if val >= 0 else -3), textcoords="offset points")
    ax.set_xticks(x, [f"{b}\n(n={n:,})" for b, n in zip(bk.index, bk["n"])])
    _style(ax, "mean next-bar return (bps)")
    _title(ax, title)
    fig.tight_layout()
    if show_fig:
        plt.show()
    return fig


def plot_equity(nets, title, show_fig=True):
    plt = _plt()
    cols = [C["s1"], C["s2"], C["s3"]]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=C["surface"])
    for (name, net), col in zip(list(nets.items())[:3], cols):
        eq = (1 + net.fillna(0)).cumprod()
        ax.plot(eq.index, eq.values, color=col, linewidth=1.6, label=name)
        ax.annotate(name, (eq.index[-1], eq.values[-1]), xytext=(4, 0), textcoords="offset points",
                    fontsize=8, color=C["ink2"], va="center")
    ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax.margins(x=0.02)
    _style(ax, "growth of 1 (log)")
    _title(ax, title)
    fig.tight_layout()
    if show_fig:
        plt.show()
    return fig


def plot_regime_timeline(daily, labels, show_fig=True):
    from matplotlib.patches import Patch
    plt = _plt()
    fig, ax = plt.subplots(figsize=(8, 3.6), facecolor=C["surface"])
    ax.plot(daily.index, daily["close"], color=C["ink2"], linewidth=1.2)
    for st, col in (("up", C["s1"]), ("down", C["red"])):
        m = (labels["dir_mom"] == st).reindex(daily.index).fillna(False).to_numpy()
        ax.fill_between(daily.index, 0, 1, where=m, transform=ax.get_xaxis_transform(),
                        color=col, alpha=0.13, linewidth=0)
    ax.set_yscale("log")
    ax.legend(handles=[Patch(color=C["s1"], alpha=0.3, label="uptrend (top third)"),
                       Patch(color=C["red"], alpha=0.3, label="downtrend (bottom third)")],
              frameon=False, fontsize=8, loc="upper left")
    ax.margins(x=0.01)
    _style(ax, "BTC close (log)")
    _title(ax, "28-day momentum regime (known at each day's close)")
    fig.tight_layout()
    if show_fig:
        plt.show()
    return fig


def summary_text(results, max_chars=14000):
    """Plain-text digest of the key tables - paste it back to Claude for interpretation."""
    parts = []
    for name, obj in results.items():
        try:
            body = obj.round(4).to_string()
        except Exception:  # non-numeric content (e.g. a data-quality Series with strings)
            body = obj.to_string() if hasattr(obj, "to_string") else str(obj)
        parts.append(f"=== {name} ===\n{body}")
    s = "\n\n".join(parts)
    return s if len(s) <= max_chars else s[:max_chars] + "\n...[truncated]"

In [ ]:
import numpy as np, pandas as pd
import btcstudy as bs

## 2. Configuration

In [ ]:
SOURCE   = "binance"      # "binance" (free, no key) | "csv" (your own file) | "synthetic" (dry run, no internet)
SYMBOL   = "BTCUSDT"
START    = "2018-01-01"   # months before Binance's first BTCUSDT file are skipped and reported
END      = None           # None = up to yesterday (UTC)
COST_BPS = 5.0            # trading cost per side in basis points (fee + slippage)
CSV_PATH = "my_data.csv"  # only for SOURCE = "csv": columns timestamp, open, high, low, close, volume

## 3. Load and check the data

In [ ]:
if SOURCE == "binance":
    raw = bs.get_data("binance", symbol=SYMBOL, interval="1m", start=START, end=END, cache_dir=CACHE)
elif SOURCE == "csv":
    raw = bs.get_data("csv", path=CSV_PATH)
else:
    raw = bs.get_data("synthetic")

quality = bs.quality_report(raw)
for k, v in quality.items():
    print(f"{k:>20}: {v}")

Check `missing_pct` (should be tiny) and `bad_ohlc_rows` (should be 0). Gaps are never filled silently.

In [ ]:
fr = bs.build_frames(raw)
df1m, h1, h4, daily = fr["1m"], fr["h1"], fr["h4"], fr["daily"]
print({k: len(v) for k, v in fr.items()})

## 4. Study A - trend or reversal, by horizon

Lag-1 autocorrelation of returns: **positive = trend, negative = reversal.** A value is only notable if it is
larger than the `2se` column (about two standard errors). The variance ratio tells the same story over longer
windows: **VR < 1 reversal, VR > 1 trend**, with a robust z-score (|z| > 2 notable).

In [ ]:
tab = bs.autocorr_by_horizon(df1m)
bs.show(tab, 4)
bs.plot_autocorr_heatmap(tab)
vr = bs.variance_ratio_table(df1m)
bs.show(vr, 3)

## 5. Study B - what happens after a big move

For each horizon: the average next-bar return after a move of z standard deviations (using only past volatility).
`reversal_bps` > 0 means the move **reversed**; `net_bps` subtracts a round-trip cost. A fade only matters if
`net_bps` is clearly positive in several buckets and several horizons.

In [ ]:
B = {}
for name, bars in {"5min": bs.resample_ohlcv(df1m, "5min"), "15min": bs.resample_ohlcv(df1m, "15min"),
                   "1h": h1, "4h": h4}.items():
    B[name] = bs.reversal_buckets(bars["close"], cost_bps=COST_BPS)
    print(f"--- {name} bars ---")
    bs.show(B[name], 2)
bs.plot_reversal(B["1h"], "After a 1h move of z standard deviations: mean next-hour return")

## 6. Study C - does volatility persist?

`abs_return_acf` positive and slowly decaying = volatility clusters. For the state rows the no-persistence
baseline is **about 0.5**: `P(high | high now)` well above 0.5 and `P(high | low now)` well below it means
today's volatility state predicts the future one.

In [ ]:
vp = pd.Series(bs.vol_persistence(daily["close"])).to_frame("value")
bs.show(vp, 3)

## 7. Study D - regimes

Labels use data up to each day's close only, so they could have been known when trading the next day.

- `dir_mom`: 28-day return in the top / bottom / middle third of its own past year (uptrend / downtrend / range)
- `er_state`: Kaufman efficiency ratio vs its past-year median; `adx_state`: ADX above 25
- `vol_state`: ATR% above / below its past-year median

**`trend_payoff_pct` > 0 means the prior direction continued over the next 5 days; < 0 means it reversed.**
`persistence` is the chance the state is the same 5 days later. The by-year table shows whether an effect holds
in every year or only in one.

In [ ]:
lab = bs.regime_labels(daily)
bs.plot_regime_timeline(daily, lab)
ev = bs.evaluate_regimes(daily, lab, horizon=5)
bs.show(ev, 3)
cons = bs.regime_consistency(daily, lab, "dir_mom", horizon=5)
bs.show(cons, 3)

## 8. Study E - simple strategies, with costs

Long-only trend rules on daily bars and 1-hour overshoot fades, all with a one-bar execution delay and
`COST_BPS` per side. Nothing is optimised: these are the literature's rules with fixed parameters.
The cost table shows how fast each one dies as costs rise; the by-year table shows consistency.

In [ ]:
suite, nets = bs.strategy_suite(daily, h1, lab, COST_BPS)
bs.show(suite, 3)
bs.plot_equity({k: nets[k] for k in ["Buy & hold", "TS momentum 28d (top third)", "Buy new 20d highs"]},
               "Growth of 1: long-only trend rules vs buy & hold")
sweep = bs.cost_sweep(daily, h1, lab)
print("Sharpe at different per-side costs:")
bs.show(sweep, 2)
daily_names = ["Buy & hold", "TS momentum 28d (top third)", "Buy new 20d highs", "Buy new 20d lows"]
by_year = pd.DataFrame({k: bs.sharpe_by_year(nets[k], 365) for k in daily_names})
print("Sharpe by calendar year:")
bs.show(by_year, 2)

### Luck test (important)

A strategy can show a good Sharpe by pure chance, especially when you try many. This test runs **the same rules
on 40 copies of the data in which every bar's return gets a random sign**: all move sizes, volatility clustering
and the average drift are kept, but any trend or reversal is destroyed.

- `p_value`: chance that a copy does at least as well. The smallest possible value is 1/41 = 0.024.
- Ten strategies are tested at once, so one of them passes by luck about 40% of the time. **Ask for
  `p_value` < 0.01 (raise `n` to 200 to reach it) and the same sign of result in every calendar year.**
- Strategies that are variations of one idea (the 10/20/50-day rules) count as one test, not three.

Calibration (60 random-walk datasets): 5.5% of strategies were flagged at the 95% level, against 5% expected.
Planted trends and reversals are detected.

In [ ]:
luck = bs.noise_comparison(h1, suite, n=40, cost_bps=COST_BPS)
bs.show(luck, 3)

## 9. Study F - your zone strategy, split by regime (optional)

Runs the zone-confluence backtest on 4-hour bars and splits its trades by the daily regime known at the previous
close: **with_trend** (long in an uptrend, short in a downtrend), **counter_trend**, or **range**.
Put `zone_agent.zip` in the same folder as this notebook. Without it the study is skipped.

In [ ]:
import zipfile
ZSYS_DIR = None   # leave None to use zone_agent.zip placed next to this notebook
if not os.path.isdir("zone_agent") and os.path.exists("zone_agent.zip"):
    zipfile.ZipFile("zone_agent.zip").extractall(".")
if os.path.isdir("zone_agent"):
    ZSYS_DIR = os.path.abspath("zone_agent")
zone = bs.zone_by_regime(df1m, lab, ZSYS_DIR) if ZSYS_DIR else None
if zone is None:
    print("Zone study skipped.")
    by_align = by_align_vol = None
else:
    zone_trades, by_align, by_align_vol = zone
    print("Zone trades by trend alignment:")
    bs.show(by_align, 3)
    print("... and by volatility state:")
    bs.show(by_align_vol, 3)

## 10. Summary to send back

In [ ]:
results = {"data quality": pd.Series({k: v for k, v in quality.items() if k != "top_gaps"}),
           "A. autocorrelation by horizon and year": tab, "A. variance ratios": vr}
for name, bk in B.items():
    results[f"B. after big moves ({name})"] = bk
results.update({"C. volatility persistence": vp, "D. regimes": ev, "D. regime trend payoff by year": cons,
                "E. strategies": suite, "E. luck test (beats_luck_95 = real Sharpe above shuffled-days 95th pct)": luck,
                "E. Sharpe vs cost": sweep, "E. Sharpe by year": by_year})
if by_align is not None:
    results["F. zone strategy by alignment"] = by_align
    results["F. zone strategy by alignment and vol"] = by_align_vol
text = bs.summary_text(results)
os.makedirs(CACHE, exist_ok=True)
open(os.path.join(CACHE, "btc_study_results.txt"), "w").write(text)
print(text)